# E06 — AGU-S · Seleção de pais preferenciais

Este notebook parte da solução salva pelo E06_01. O projeto separado guarda a configuração final, o resultado e o inventário de equipamentos com suas cargas. O resultado inclui os pais preferenciais e alternativos, as métricas de segmento e os ranks anunciados.

## 1. Importar a solução e inspecionar a rede de referência

Depois de salvar a solução final no E06_01, importe-a para o projeto de destino com `import_solution()`. Se a cópia já existir, o SDK usa os arquivos locais. Em seguida, abra o novo projeto com `open_project()`, carregue o `result.json` e visualize a solução pela sessão do planejador mesh.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

source_project = PlanningProject("../projects/e06_agu_redesign")
project = PlanningProject("../projects/e06_agu_parent_selection")
project.import_solution(source_project)

client = PlanningClient("http://network-service:8080", workspace="..")
scenario = client.open_project(project, planner="mesh")
baseline = scenario.load_result()
scenario.show_mesh_network()

## 2. Consultar a nota da rede de referência

A nota e o resumo são os valores gravados pelo E06_01. Eles serão a referência para o teste de seleção de pais. O critério discutido é minimizar a nota da rede por trocas dentro do conjunto de pais mantidos, preservando o rank anunciado.

In [ ]:
baseline.get("network_evaluation", {})

## 3. Reduzir o número médio de saltos

A métrica `hop_count_path` conta os saltos de cada rádio com equipamento associado até a raiz. O TOML `default_network_metric.toml` calcula a média dessas notas. Assim, encurtar um caminho pode melhorar a nota global mesmo quando o caminho mais longo permanece igual.

A média pode aceitar que alguns caminhos fiquem mais longos se a melhora nos demais compensar. Este teste usa os pais elegíveis da solução importada. O mapa seguinte mostra os caminhos da rede resultante.

In [ ]:
baseline = scenario.load_result()
hops_test = scenario.optimize_parents(path_metric="hop_count_path", network_metric="default_network")
{"initial_score": hops_test["baseline"]["network_score"],
 "final_score": hops_test["optimized"]["network_score"],
 "changes": hops_test["changes"]}

In [ ]:
scenario.show_mesh_network()

## 4. Reduzir o número de enlaces congestionados

A pressão de um enlace é o custo do segmento multiplicado pela carga total que passa por ele. A nota da rede conta os enlaces distintos com pressão acima de 125, valor inicial de teste definido no TOML. A nota de cada caminho mostra seu gargalo. Este teste começa de novo pela rede importada. O resumo apresenta as notas inicial e final e as trocas aceitas. A célula seguinte mostra a rede da sessão após o teste.

O TOML `congested_links_network_metric.toml` conta uma vez cada enlace com `carga total × custo do segmento` acima de 125. Esse é o limiar experimental configurado para o teste. O mapa abre com todos os caminhos usados por rádios com equipamentos associados. Repetidores e acessos podem aparecer como trechos intermediários ou destinos.

In [ ]:
baseline = scenario.load_result()
congestion_test = scenario.optimize_parents(path_metric="max_load_weighted_path", network_metric="congested_links_network")
{"initial_score": congestion_test["baseline"]["network_score"],
 "final_score": congestion_test["optimized"]["network_score"],
 "changes": congestion_test["changes"]}

In [ ]:
scenario.show_mesh_network()